# O*NET ElasticNet Tables

Batch export table-style images from `../../result/occ/analysis/lasso_enet_results.csv` to `../../result/occ/LASSO`.

In [1]:
from pathlib import Path
import textwrap

import matplotlib.pyplot as plt
import pandas as pd

BASE_DIR = Path.cwd().resolve().parents[1]
INPUT_CSV = BASE_DIR / "result" / "occ" / "analysis" / "lasso_enet_results.csv"
OUTPUT_DIR = BASE_DIR / "result" / "occ" / "LASSO"

PANEL_LABELS = {
    "Work Activities": "Panel A: Work Activities",
    "Knowledge": "Panel B: Knowledge",
    "Skills": "Panel C: Skills",
}

TABLE_SPECS = [
    {
        "outcome": "unemployment",
        "tables": ["Work Activities", "Knowledge", "Skills"],
        "title": "ElasticNet Selected O*NET Features for Unemployment",
        "filename": "unemployment_elasticnet_table.png",
    },
    {
        "outcome": "employment",
        "tables": ["Work Activities", "Knowledge", "Skills"],
        "title": "ElasticNet Selected O*NET Features for Employment",
        "filename": "employment_elasticnet_table.png",
    },
   
    {
        "outcome": "hours",
        "tables": ["Work Activities", "Knowledge", "Skills"],
        "title": "ElasticNet Selected O*NET Features for Hours",
        "filename": "hours_skills_elasticnet_table.png",
    },
    {
        "outcome": "income",
        "tables": ["Work Activities", "Knowledge", "Skills"],
        "title": "ElasticNet Selected O*NET Features for Income",
        "filename": "income_elasticnet_table.png",
    },
    {
        "outcome": "hourly_rate",
        "tables": ["Knowledge"],
        "title": "ElasticNet Selected O*NET Features for Hourly Rate",
        "filename": "hourly_rate_elasticnet_table.png",
    },
    {
        "outcome": "income_share",
        "tables": ["Work Activities", "Knowledge", "Skills"],
        "title": "ElasticNet Selected O*NET Features for Income Share",
        "filename": "income_share_elasticnet_table.png",
    },
    {
        "outcome": "inequality",
        "tables": ["Work Activities", "Knowledge", "Skills"],
        "title": "ElasticNet Selected O*NET Features for inequality",
        "filename": "inequality_elasticnet_table.png",
    }
    
]

def split_pipe_list(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [item.strip() for item in str(value).split("|") if item.strip()]

def wrap_feature(feature, width=52):
    return textwrap.wrap(feature, width=width, break_long_words=False, break_on_hyphens=False) or [str(feature)]

def load_results():
    return pd.read_csv(INPUT_CSV)

def build_panels(df, outcome, tables):
    subset = df[(df["outcome"] == outcome) & (df["method"] == "ElasticNet") & (df["table"].isin(tables))].copy()
    subset["table"] = pd.Categorical(subset["table"], categories=tables, ordered=True)
    subset = subset.sort_values("table")

    if subset.empty:
        raise ValueError(f"No rows found for outcome='{outcome}' and method='ElasticNet'.")

    panels = []
    for table_name in tables:
        row = subset[subset["table"] == table_name]
        if row.empty:
            continue

        row = row.iloc[0]
        features = split_pipe_list(row["top10_features"])
        coefs = [float(item) for item in split_pipe_list(row["top10_coefs"])]

        panels.append({
            "table": table_name,
            "title": PANEL_LABELS[table_name],
            "items": list(zip(features, coefs)),
            "adj_r2": float(row["r_squared_adj"]),
        })

    if not panels:
        raise ValueError(f"No matching tables found for outcome='{outcome}'.")

    return panels

def count_rows(title, panels):
    total = 0
    total += 2
    for panel in panels:
        total += 1
        for feature, _ in panel["items"]:
            total += len(wrap_feature(feature))
        total += 1
        total += 1
    return total + 1

def render_table(title, panels, output_path):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    plt.rcParams["font.family"] = "STIXGeneral"
    plt.rcParams["figure.dpi"] = 220

    total_rows = count_rows(title, panels)
    fig_w = 8.8
    fig_h = max(3.8, total_rows * 0.34)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.set_xlim(0, 1)
    ax.set_ylim(total_rows, 0)
    ax.axis("off")

    left_x = 0.03
    coef_x = 0.965
    y = 1.0

    ax.text(left_x, y, title, ha="left", va="center", fontsize=16, fontweight="bold")
    y += 0.5
    ax.hlines(y, left_x, coef_x, colors="black", linewidth=0.8)
    y += 0.9

    for idx, panel in enumerate(panels):
        ax.text(left_x, y, panel["title"], ha="left", va="center", fontsize=14, fontweight="bold")
        y += 1.0

        for feature, coef in panel["items"]:
            wrapped = wrap_feature(feature)
            for line_idx, line in enumerate(wrapped):
                ax.text(left_x, y, line, ha="left", va="center", fontsize=12.5)
                if line_idx == 0:
                    ax.text(coef_x, y, f"{coef:.3f}", ha="right", va="center", fontsize=12.5)
                y += 1.0

        ax.text(left_x, y, "Adjusted R-squared", ha="left", va="center", fontsize=12.5, fontweight="bold")
        ax.text(coef_x, y, f"{panel['adj_r2']:.3f}", ha="right", va="center", fontsize=12.5, fontweight="bold")
        y += 0.9

        ax.hlines(y, left_x, coef_x, colors="black", linewidth=0.5 if idx < len(panels) - 1 else 0.8)
        y += 0.9

    fig.savefig(output_path, bbox_inches="tight", facecolor="white")
    plt.close(fig)

def generate_all_tables():
    df = load_results()
    summary_rows = []

    for spec in TABLE_SPECS:
        panels = build_panels(df, spec["outcome"], spec["tables"])
        output_path = OUTPUT_DIR / spec["filename"]
        render_table(spec["title"], panels, output_path)
        summary_rows.append({
            "outcome": spec["outcome"],
            "tables": " | ".join(spec["tables"]),
            "n_panels": len(panels),
            "output_file": str(output_path),
        })

    return pd.DataFrame(summary_rows)

summary_df = generate_all_tables()
summary_df


/tmp/ipykernel_21811/4180744548.py:77: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  subset["table"] = pd.Categorical(subset["table"], categories=tables, ordered=True)


,outcome,tables,n_panels,output_file
0,unemployment,Work Activities | Knowledge | Skills,3,/home/banjiayu/workspace/Thesis-code/result/oc...
1,employment,Work Activities | Knowledge | Skills,3,/home/banjiayu/workspace/Thesis-code/result/oc...
2,hours,Work Activities | Knowledge | Skills,3,/home/banjiayu/workspace/Thesis-code/result/oc...
3,income,Work Activities | Knowledge | Skills,3,/home/banjiayu/workspace/Thesis-code/result/oc...
4,hourly_rate,Knowledge,1,/home/banjiayu/workspace/Thesis-code/result/oc...
5,income_share,Work Activities | Knowledge | Skills,3,/home/banjiayu/workspace/Thesis-code/result/oc...
6,inequality,Work Activities | Knowledge | Skills,3,/home/banjiayu/workspace/Thesis-code/result/oc...


In [ ]:
FEATURE_SUMMARY_XLSX = OUTPUT_DIR / "elasticnet_repeated_features_summary.xlsx"
MIN_OCCURRENCES = 2

def build_repeated_feature_tables(table_specs, min_occurrences=2):
    df = load_results()
    selected_rows = []
    outcome_order = [spec["outcome"] for spec in table_specs]

    for spec in table_specs:
        subset = df[
            (df["outcome"] == spec["outcome"])
            & (df["method"] == "ElasticNet")
            & (df["table"].isin(spec["tables"]))
        ].copy()

        for _, row in subset.iterrows():
            features = split_pipe_list(row["top10_features"])
            coefs = [float(item) for item in split_pipe_list(row["top10_coefs"])]

            for feature, coef in zip(features, coefs):
                selected_rows.append({
                    "table": row["table"],
                    "element_name": feature,
                    "outcome": row["outcome"],
                    "coefficient": coef,
                })

    detail_df = pd.DataFrame(selected_rows)
    if detail_df.empty:
        raise ValueError("No ElasticNet features found for the configured TABLE_SPECS.")

    detail_df["outcome"] = pd.Categorical(detail_df["outcome"], categories=outcome_order, ordered=True)
    detail_df = detail_df.sort_values(["table", "element_name", "outcome"]).reset_index(drop=True)

    summary_rows = []
    for (table_name, element_name), group in detail_df.groupby(["table", "element_name"], sort=True):
        group = group.sort_values("outcome")
        occurrence_count = group["outcome"].nunique()
        if occurrence_count < min_occurrences:
            continue

        outcomes = [str(outcome) for outcome in group["outcome"].tolist()]
        coefficient_pairs = [f"{outcome}: {coef:.6f}" for outcome, coef in zip(outcomes, group["coefficient"].tolist())]

        row = {
            "table": table_name,
            "element_name": element_name,
            "occurrence_count": occurrence_count,
            "outcomes": " | ".join(outcomes),
            "coefficients": " | ".join(coefficient_pairs),
        }

        coef_map = {str(outcome): coef for outcome, coef in zip(group["outcome"], group["coefficient"])}
        for outcome in outcome_order:
            row[f"coef_{outcome}"] = coef_map.get(outcome)

        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    if summary_df.empty:
        raise ValueError(f"No repeated features found with min_occurrences={min_occurrences}.")

    summary_df = summary_df.sort_values(["table", "occurrence_count", "element_name"], ascending=[True, False, True]).reset_index(drop=True)
    return summary_df, detail_df

repeated_feature_df, repeated_feature_detail_df = build_repeated_feature_tables(TABLE_SPECS, min_occurrences=MIN_OCCURRENCES)

with pd.ExcelWriter(FEATURE_SUMMARY_XLSX) as writer:
    repeated_feature_df.to_excel(writer, sheet_name="feature_summary", index=False)
    repeated_feature_detail_df.to_excel(writer, sheet_name="feature_detail", index=False)

print(f"Saved repeated-feature summary to: {FEATURE_SUMMARY_XLSX}")
repeated_feature_df
